# 22 · 做成服务：把 agent 变成别人能调用的接口，再装进容器

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → [22 做成服务 ✅]
```

> **这一章的代码在合并请求 #24 里。** 开发这一章时一共真问了大约 13 次模型，约 ¥0.15。运行这本笔记：第 3 步真问 1 次（约 ¥0.02），其余全部在本机计算，不花钱；把"准备"里的 `ASK_THE_MODEL` 改成 `False` 就一分钱不花。
>
> 第 2、4 步会在本机启动服务（端口 8022 和 8023），最后一格把它们停掉。重复运行是安全的：每次启动前会先停掉上一次的。

## 背景：会答题，和能给别人用，是两回事

到第 21 章，agent 只有一种用法：在终端里敲 `raredx eval run`，它从文件里读出 840 道题，一道道答完，把分数记进 MLflow。这是**考试**的用法。

真正要用它的不是我们自己。是医院的系统、别的团队的程序、一个网页。他们的用法是：发来**一个**病人，等十来秒，要回一个答案。而且：

- 他们不装我们的 Python 环境，也不敲我们的命令
- 他们发来的东西可能是错的，比如打错一个编号
- DeepSeek 可能正好出故障。这不是假设：第 20 章大考考到一半，账户就欠费过一次
- 事后有人问"这个答案是怎么来的"，要能查

打个比方：前 21 章是**在后厨把菜做好，还请人打了分**。这一章是**开门营业**。开门要有菜单（接口说明）、固定格式的点菜单（请求格式）、煤气断了时的预案（降级）、每一桌的小票（记录），还要做到"整套搬去另一个店面，照样能开"（容器）。

**公司里的做法**：模型或 agent 评测完以后，交付出去的东西几乎都是"一个 HTTP 服务，加一个容器镜像"。别的团队只看接口说明，不看你的代码。招聘要求里的 serving、API design、Docker、productionize，说的就是这一章。

## 贯穿这一章的一条原则：考的是什么，上线的就是什么

第 21 章花了约 ¥39，才得到"42.4%"这个能对外说的数字。如果为了做服务去改 agent 的代码，上线的就是另一个没考过的系统，这个数字就不能再用了。

所以这一章**没有改 agent 的一个字**（第 2 步会验证）。服务只做两件事：把请求原样交给考试时的那个 agent；在它周围加上接口、检查、记录和兜底。

## 这一章做的 8 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 定接口：怎么问，怎么答 | FastAPI + pydantic，自动生成接口说明 | 别人不看代码也知道怎么用；乱发的东西进不来 |
| 2 | 启动服务 | uvicorn，`raredx serve`，健康检查 | 任何能发 HTTP 请求的程序都能来问 |
| 3 | 实时看 agent 的每一步 | 服务器推送事件（SSE），LangChain 回调 | 等的十几秒里，看得见它在查什么 |
| 4 | 模型出故障时自动降级 | 事先算好一份不用模型的答案备用 | 用户仍然拿得到答案，并且知道是降级的 |
| 5 | 把门 | 输入校验、并发上限、密钥、截止时间 | 不白花钱，不被拖垮，不悄悄答错 |
| 6 | 每次请求留记录 | 一条 MLflow 追踪 + 一行日志 | 事后查得清；病人的症状不进日志 |
| 7 | 演示页面 | Streamlit | 不会写程序的人也能试 |
| 8 | 装进容器，CI 每次检查 | Docker、Compose、GitHub Actions | 换台电脑一条命令跑起来；坏的镜像合并不进去 |

---

## 准备

运行下面两格。第一格定义一个叫 `run` 的小工具，用来执行命令并显示结果；第二格定义启动、停止服务和发请求的小工具。**不用看代码。**

第二格开头有一个开关 `ASK_THE_MODEL`：第 3 步会真的问一次大模型（约 ¥0.02）。改成 `False`，这本笔记就完全不花钱。

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

In [ ]:
import json, logging, tempfile, time, urllib.error, urllib.request, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)

ASK_THE_MODEL = True   # 第 3 步真问一次模型，约 ¥0.02。改成 False：这本笔记完全不花钱

LOGS = Path(tempfile.mkdtemp(prefix="raredx-serve-"))   # 服务的日志放在临时文件夹，不进项目
SERVERS = {}

def stop(port):
    """停掉这个端口上的服务（包括上次没关干净的）。"""
    p = SERVERS.pop(port, None)
    if p is not None:
        p.terminate()
        try:
            p.wait(timeout=20)
        except subprocess.TimeoutExpired:
            p.kill()
    # 只找“守着这个端口”的进程（LISTEN）。连着它的客户端不算：这本笔记自己就是一个
    subprocess.run(f"lsof -ti tcp:{port} -sTCP:LISTEN | xargs kill 2>/dev/null", shell=True)

def start(port, with_model=True):
    """在这个端口启动服务，等它准备好。with_model=False：不给模型密钥。"""
    stop(port)
    time.sleep(0.5)
    env = dict(os.environ)
    if not with_model:
        env["RAREDX_LLM_API_KEY"] = ""   # 空的密钥 = 没有模型
    print(f"$ uv run raredx serve --port {port}" + ("" if with_model else "      （这一次不给模型密钥）"))
    log = open(LOGS / f"{port}.log", "w")
    SERVERS[port] = subprocess.Popen(["uv", "run", "raredx", "serve", "--port", str(port)],
                                     env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(240):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/readyz", timeout=1)
            print(f"服务已经在 http://127.0.0.1:{port} 等着了")
            return
        except Exception:
            if SERVERS[port].poll() is not None:
                raise RuntimeError((LOGS / f"{port}.log").read_text()[-2000:])
            time.sleep(0.5)
    raise RuntimeError("两分钟还没准备好")

def ask(url, body=None, headers=None):
    """发一个 HTTP 请求，返回（状态码，回答）。给了 body 就是 POST。"""
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(url, data=data, headers={"content-type": "application/json", **(headers or {})})
    try:
        with urllib.request.urlopen(request, timeout=180) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())

print("小工具准备好了。真问模型：", "是（第 3 步，约 ¥0.02）" if ASK_THE_MODEL else "否（完全不花钱）")

---

## 第 1 步 · 定接口：别人怎么问，我们怎么答

### 在干什么
把"一个请求长什么样、一个回答长什么样"规定下来。规定是写成代码的，不是写成文档的。

### 背景
考试时，题目是我们自己的程序从文件里读出来的，格式一定对。服务不一样，输入是外面发来的：可能少一项，可能多一项，可能编号打错。不在门口检查的话，错误会在很深的地方才冒出来；更糟的情况是不报错，悄悄给一个错的答案。

### 用了什么技术，为什么

| 技术 | 是什么 | 打个比方 |
|---|---|---|
| **FastAPI** | Python 里做 HTTP 接口最常用的框架 | 餐厅前台：接单，递给后厨，把菜端出去 |
| **pydantic** | 用代码描述"一份数据应该长什么样"，不符合就拒绝 | 点菜单：上面只有可以打勾的格子 |
| **OpenAPI 文档** | FastAPI 根据上面的描述自动生成的接口说明网页，地址是 `/docs` | 自动印出来的菜单，和后厨实际能做的永远一致 |

**一个请求只有四项**（代码在 `src/raredx/service/schemas.py`）：

| 项 | 内容 | 格式要求 |
|---|---|---|
| `hpo_ids` | 病人的表现，HPO 编号 | 1 到 100 个，每个必须是 `HP:` 加 7 位数字 |
| `sex` | 性别 | 只能是 `female`、`male`、`unknown` |
| `candidate_genes` | 测序后剩下的候选基因（可以不给） | 最多 200 个基因符号 |
| `exclude_sources` | 要从知识库里遮掉的论文（可以不给） | `PMID:` 加数字 |

**没有任何一项可以随便写字。** 这是有意的：名字、生日、病历原文，根本发不进来。发不进来的东西，就不可能出现在日志里、追踪记录里，或者发给 DeepSeek 的内容里。行业里管这个叫**数据最小化**（data minimisation）：用不着的信息，在入口就不收。

**一个回答里有什么**：

| 项 | 内容 |
|---|---|
| `mode` | 这个答案是 agent 给的（`agent`），还是降级给的（`fallback`） |
| `diagnoses` | 最多 10 个病，每个带：名次、agent 的理由、**证据**（病人的哪些表现它解释得了、哪些解释不了、多常见、出处是哪篇文献） |
| `steps` | 它一步步做了什么 |
| `usage` | 用了几秒、调了几次模型、多少 token、大约多少钱 |
| `trace_id` | 这次请求的完整记录在 MLflow 里的编号（第 6 步） |
| `system` | 答题的是哪一版：模型、提示词版本、工具、知识库版本、代码版本 |
| `disclaimer` | 一句固定的话：这是研究软件，不是医疗器械 |

**为什么选 FastAPI**：Flask 更老牌，但格式检查和接口文档都要自己另外写；Django 是做整个网站的，太重。FastAPI 把"格式检查、自动文档、分段输出"都带了，是现在 Python 里做模型服务的事实标准。

### 公司里什么情况怎么做
- **接口先行**：前端、后端、别的团队先把接口格式定下来，再各自开发，互不等待
- **接口带版本号**：我们的地址都以 `/v1` 开头。以后格式要改，就出 `/v2`，老用户不受影响
- **医疗数据在入口就限制**：能不收的不收，比收进来再保护要可靠得多

### 执行
不用启动服务，直接看这份"点菜单"会接受什么、拒绝什么：

In [ ]:
from pydantic import ValidationError
from raredx.service.schemas import DiagnoseRequest

print("请求里允许出现的项：", list(DiagnoseRequest.model_fields))

ok = DiagnoseRequest(hpo_ids=["HP:0001083", "HP:0001166", "HP:0002616"], sex="male")
print("\n一个合格的请求：", ok.model_dump_json(exclude_defaults=True))

bad = [
    ("写的是文字，不是编号", {"hpo_ids": ["晶状体脱位"]}),
    ("多了一项（想顺手发个名字）", {"hpo_ids": ["HP:0001083"], "name": "张三"}),
    ("一个表现都没有", {"hpo_ids": []}),
    ("性别写了别的", {"hpo_ids": ["HP:0001083"], "sex": "男"}),
]
for label, body in bad:
    try:
        DiagnoseRequest.model_validate(body)
    except ValidationError as e:
        print(f"\n{label}\n   → 拒绝：{e.errors()[0]['msg']}")

### 结果怎么看
- 允许的项只有四个：`hpo_ids`、`sex`、`candidate_genes`、`exclude_sources`
- 合格的请求就是一小段 JSON。那三个编号是晶状体脱位、蜘蛛指、主动脉根部瘤，马凡综合征的经典三联征，后面还会用到
- 四个不合格的请求都被拒绝，并且说了原因：
  - 文字不是编号 → `String should match pattern '^HP:\d{7}$'`
  - 多了 `name` 这一项 → `Extra inputs are not permitted`（不接受多余的项）。**名字就是这样被挡在外面的**
  - 没有表现 → `List should have at least 1 item`
  - 性别写"男" → 只接受 `'female', 'male' or 'unknown'`

服务跑起来以后，浏览器打开 `http://127.0.0.1:8022/docs` 能看到自动生成的接口说明，可以直接在网页上试着发请求。

### 面试怎么说
> "The request schema accepts coded data only — HPO ids, sex, gene symbols, PubMed ids, each pattern-checked, with extra fields forbidden. There is no free-text field, so a name or a date of birth cannot reach a log, a trace or the model provider by mistake. The schema is also the contract: FastAPI generates the OpenAPI reference from it."

---

## 第 2 步 · 启动服务：它活着吗，上的是哪一版

### 在干什么
用一条命令把服务跑起来，然后用最普通的 HTTP 请求问它三件事：你活着吗？你能接客了吗？你现在是哪一版？

### 背景
以前要用 agent，必须装好我们的 Python 环境，在项目目录里敲命令。服务跑起来以后，任何能发 HTTP 请求的东西都能来问：浏览器、Java 程序、医院的系统、命令行里的 `curl`。

### 用了什么技术，为什么

| 技术 | 是什么 | 打个比方 |
|---|---|---|
| **uvicorn** | 真正守着端口、收发网络数据的程序。FastAPI 管"收到以后怎么处理"，uvicorn 管"怎么收发" | 餐厅的门和服务员 |
| **`/healthz`**（存活检查） | 进程还在不在 | 店里有没有人 |
| **`/readyz`**（就绪检查） | 能不能回答问题了：知识库加载完了没有 | 可以点菜了没有 |
| **`/v1/system`** | 现在答题的是哪一版 | 墙上挂的营业执照 |

**启动时它做了什么**：读 HPO（约 4 秒），读病例库的 6,014 个已发表病人，建好两个 agent，然后才开始接请求。这些大件只在启动时加载一次，不是每来一个请求加载一次。

**为什么是两个 agent**：考试时就是两个（第 21 章）。请求按有没有候选基因，交给考过的那一个：

| 请求 | 交给谁 | 它在留出集上的成绩 |
|---|---|---|
| 没有候选基因 | 只看症状的 agent（5 个工具） | 第 1 名 42.4%，前 10 名 62.0% |
| 有候选基因 | 多一个基因工具的 agent，答案再和基因排名合并 | 第 1 名 65.9%，前 10 名 81.4% |

**病例库用哪些病人**：默认只用考试时用的那一半（6,014 人）。全部 10,374 人都放进去，能查到的病人更多，但那个配置没有考过，分数不知道，所以默认不开（设置 `RAREDX_SERVICE_PATIENT_BANK=all` 可以打开）。

**为什么健康检查不顺便问一下模型**：两个原因。问一次要花钱，而健康检查每隔几秒就来一次。更重要的是，模型挂了我们照样能回答（第 4 步）；如果因为模型挂了就报告"不健康"，管理容器的系统会把我们重启，反而连降级的答案都给不出了。

### 公司里什么情况怎么做
- **Kubernetes 靠这两个检查做决定**：不存活就重启；没就绪就先不把用户的请求分过来
- **一个服务开很多份**（副本）来接更多请求。我们这个每份占 300 多 MB 内存（实测 322 MB），启动约 5 秒
- **每个回答都带版本信息**：出了问题第一句话就是"哪一版答的"

### 执行
启动服务（带模型密钥，从 `.env` 里读），问前两个问题：

In [ ]:
start(8022)
run("curl -s http://127.0.0.1:8022/healthz")
run("curl -s http://127.0.0.1:8022/readyz")

第三个问题：现在是哪一版在答题？

In [ ]:
_, system = ask("http://127.0.0.1:8022/v1/system")
print("服务版本：", system["service_version"], "   代码版本：", system["code"])
print("\n没有候选基因的请求，交给这个 agent：")
for key in ("model", "prompt_version", "max_model_calls", "tools", "patient_bank"):
    print(f"   {key:<16} {system['agent'][key]}")
print("\n有候选基因的请求，交给这个 agent：")
print(f"   {'tools':<16} {system['agent_with_genes']['tools']}")
print(f"   {'fusion':<16} {system['agent_with_genes']['fusion']}")
print("\n模型用不了时由谁回答：", system["fallback"]["without_candidate_genes"], "（没有基因） /",
      system["fallback"]["with_candidate_genes"], "（有基因）")
print("限制：", system["limits"])

最后验证那条原则：从考试的版本 `v1.0` 到现在，agent、工具、对照系统、防泄露这几部分的代码有没有改过？

In [ ]:
run("git diff --stat v1.0 -- src/raredx/agent.py src/raredx/tools src/raredx/baselines src/raredx/knowledge/masking.py; "
    "echo '（上面没有任何输出 = 这些文件从 v1.0 到现在一个字没改）'")

### 结果怎么看
- `/healthz` 回 `{"status":"ok"}`；`/readyz` 回 `{"status":"ready","hpo_release":"2026-09-01","model_available":true}`：知识库是 2026-09-01 那一版，模型可用
- `/v1/system` 里的内容和第 21 章考试记录的参数是同一套：模型 `deepseek-flash`，提示词 `seeded-v1`，最多 8 次模型调用，5 个工具，病例库 `design: 6014 patients`
- 带基因的那个 agent 多一个工具 `rank_candidate_genes`，并且写着 `reciprocal rank (k=60)`，就是第 20 章加的"和基因排名合并"
- `git diff` 没有任何输出。**服务调用的 agent 代码，和花 ¥39 考过的那一份，一个字不差**

有三处和考试不完全一样，如实列出来：单次模型调用最多等 60 秒（考试时是 180 秒）；请求里不给性别时，写的是 `UNKNOWN_SEX`（和病例库里大多数病人的写法一样）；大模型本身同一道题两次回答可能略有不同。这三处都不改变系统的做法。

### 面试怎么说
> "The service routes a request to the configuration that was evaluated for that kind of input — the phenotype-track agent, or the genotype-track agent when candidate genes are sent — over the same patient bank. `git diff v1.0` on the agent, tools and masking code is empty: what answers is what was measured. Liveness and readiness are separate, and neither calls the model."

---

## 第 3 步 · 实时看 agent 的每一步（这一步真问一次模型）

### 在干什么
发一个真病人过去。答案出来之前，一步一步看它在做什么。

### 背景
agent 回答一次要十秒左右。对着一个转圈的图标等十秒，用户会以为卡死了。更要紧的是：医生要看到它**查了什么、凭什么这么判断**，才敢参考它的答案。只给一个病名，没人敢信。

### 用了什么技术，为什么

| 技术 | 是什么 | 打个比方 |
|---|---|---|
| **SSE**（Server-Sent Events，服务器推送事件） | 一次请求，服务器分很多次把内容发回来 | 外卖软件里的"商家已接单 → 骑手已取餐 → 还有 500 米" |
| **LangChain 回调**（callback） | agent 每调一次模型、每用一次工具，就通知一下登记过的"旁听者" | 在后厨开一个传菜窗口：厨师照常做菜，外面看得见做到哪一步了 |

ChatGPT 的字一个个蹦出来，用的就是 SSE。我们推送的不是一个个字，是一步步动作。

**关键的设计：不改 agent，从外面看。** 如果为了"能被看见"去改 `agent.py`，上线的就不是考过的那一份了。LangChain 自带一个机制：把"旁听者"放在一个上下文变量里，在这个上下文里启动的 agent 会自动带上它（LangChain 自己统计 token 用量的计数器就是这么挂上去的）。所以服务只是在调用 agent 之前挂上旁听者，调用完摘下来。代码在 `src/raredx/service/watch.py`。

**"旁听者"还兼管刹车。** 每次调用模型之前，它检查两件事：过了截止时间吗？用户是不是已经关掉页面走了？有一条成立，就在这两步之间停下来。花钱的是下一次模型调用，在它前面停，正好不多花。

**用哪个病人**：示例 2，一个已经发表的女性病人（PMID:25681079），8 个表现：红细胞里一种酶（GALT）活性降低、血里半乳糖高、黄疸、呕吐、喂养困难、肌张力低、癫痫发作、语言发育迟缓。论文确诊的是**半乳糖血症**（GALT 基因）。这道题在留出集里，考试时的情况是：传统排名的前 10 名里没有它，agent 把它排在第 1。

请求里带了 `exclude_sources: ["PMID:25681079"]`：把她自己那篇论文从知识库里遮掉，和考试时一样。不遮的话，答案等于是从她自己的论文里"读"出来的，不算本事。

### 公司里什么情况怎么做
- **凡是要等几秒以上的 AI 功能，基本都分段输出**
- **agent 产品把中间步骤展示出来**，是为了让人信得过，也方便用户发现它走偏了
- **用户中途走了要能停**：行业里叫协作式取消（cooperative cancellation），在步骤之间检查，不硬杀线程

### 执行
先看这个病人和发出去的请求（不花钱）：

In [ ]:
from raredx.service.client import Client
from raredx.service.schemas import Step

api = Client("http://127.0.0.1:8022")
example = api.examples()[1]
print(f"病人：{example.source}，{len(example.findings)} 个表现")
for f in example.findings:
    print(f"   - {f.label}  {f.hpo_id}")
print("\n发出去的请求：")
print(example.request.model_dump_json(exclude_defaults=True, indent=1))

发出去，边等边看（约 10 秒，约 ¥0.02）：

In [ ]:
KIND = {"tools": "工具", "model": "模型", "tool": "工具", "answer": "答案", "fallback": "降级"}
AGENT_ANSWER = None
if not ASK_THE_MODEL:
    print("ASK_THE_MODEL = False：这一格跳过，不问模型。下面“结果怎么看”里有我运行时的真实输出。")
else:
    for item in api.stream(example.request):
        if isinstance(item, Step):
            print(f"{item.seconds:5.1f} 秒  [{KIND[item.kind]}] {item.title}")
            if item.detail:
                print(f"                  {item.detail}")
        else:
            AGENT_ANSWER = item
    u = AGENT_ANSWER.usage
    print(f"\n回答方式：{AGENT_ANSWER.mode}    用时 {u.seconds} 秒    调用模型 {u.model_calls} 次    "
          f"{u.input_tokens + u.output_tokens:,} 个 token（其中 {u.cached_input_tokens:,} 个命中缓存）    约 ${u.cost_usd_estimate:.4f}")
    print("\n前 3 名：")
    for d in AGENT_ANSWER.diagnoses[:3]:
        matched = f"对上 {len(d.evidence.match.explained)}/{d.evidence.match.n_findings} 个表现" if d.evidence else "HPO 里没有它的资料"
        print(f"   {d.rank}. {d.disease_name}（{d.disease_id}，基因 {d.gene}）   {matched}")
    rank = next((d.rank for d in AGENT_ANSWER.diagnoses if d.disease_id == example.confirmed_disease_id), None)
    print(f"\n论文确诊的病：{example.confirmed_disease_name}（{example.confirmed_disease_id}）→ "
          + (f"这次排在第 {rank} 名" if rank else "这次不在前 10 名里"))
    print("\nagent 给第 1 名的理由：")
    print("  ", AGENT_ANSWER.diagnoses[0].reason)

### 结果怎么看
大模型每次的做法会略有不同，所以你看到的步骤数和措辞可能和下面不完全一样。我有一次运行时是这样的：

```
  0.4 秒  [工具] Phenotype tools: 8,478 diseases ranked, the best 30 checked against the patient
                  Leading: 3-Methylcrotonyl-CoA carboxylase 1 deficiency (6/8 findings); Galactose epimerase deficiency (5/8 findings); Mitochondrial DNA depletion syndrome 3 (hepatocerebral type) (5/8 findings)
  1.8 秒  [模型] Model, turn 1: asks the tools
                  find_disease, find_similar_patients
  1.9 秒  [工具] find_disease: “Galactosemia classic”
                  Closest: Galactosemia (OMIM:230400); Ehlers-Danlos syndrome, classic type, 2 (OMIM:130010); Ehlers-Danlos syndrome, classic-like (OMIM:606408)
  1.9 秒  [工具] find_similar_patients: published patients like this one
                  10 closest of 6,014 published patients: Mitochondrial DNA depletion syndrome 6 (hepatocerebral type) ×4; Diamond-Blackfan anemia 1 ×2; HMG-CoA synthase-2 deficiency ×2
  3.3 秒  [模型] Model, turn 2: asks the tools
                  check_match, lookup_disease
  3.3 秒  [工具] check_match: Galactosemia
                  explains 1 of 8 findings
  3.3 秒  [工具] lookup_disease: Galactosemia
                  22 features; genes GALT
  4.7 秒  [模型] Model, turn 3: asks the tools
                  check_match, find_disease
  4.7 秒  [工具] find_disease: “Galactosemia type 2”
                  Closest: Galactosemia (OMIM:230400); Acrofacial dysostosis, Catania type (OMIM:101805); Keratoderma, palmoplantar, punctate type 3 (OMIM:101850)
  4.8 秒  [工具] check_match: Mitochondrial DNA depletion syndrome 6 (hepatocerebral type)
                  explains 5 of 8 findings
  5.9 秒  [模型] Model, turn 4: asks the tools
                  find_disease ×2
  6.0 秒  [工具] find_disease: “Galactosemia, Duarte variant”
                  Closest: Galactosemia (OMIM:230400); Ohdo syndrome, Say-Barber-Biesecker-Young-Simpson variant (OMIM:603736); Ceroid lipofuscinosis, neuronal, 8, northern epilepsy variant (OMIM:610003)
  6.1 秒  [工具] find_disease: “GALT deficiency”
                  Closest: Adenylosuccinase deficiency (OMIM:103050); Ventricular arrhythmias due to cardiac ryanodine receptor calcium release deficiency syndrome (OMIM:115000); Leukocyte adhesion deficiency, type I (OMIM:116920)
 10.5 秒  [模型] Model, turn 5: gives its answer
                  10 diagnoses, first: Galactosemia (classic, GALT deficiency)
 10.5 秒  [答案] Answer: 10 diagnoses, first: Galactosemia (classic, GALT deficiency)

回答方式：agent    用时 10.61 秒    调用模型 5 次    28,934 个 token（其中 26,624 个命中缓存）    约 $0.0020

前 3 名：
   1. Galactosemia（OMIM:230400，基因 GALT）   对上 1/8 个表现
   2. Galactose epimerase deficiency（OMIM:230350，基因 GALE）   对上 5/8 个表现
   3. Galactokinase deficiency with cataracts（OMIM:230200，基因 GALK1）   对上 2/8 个表现

论文确诊的病：Galactosemia（OMIM:230400）→ 这次排在第 1 名

agent 给第 1 名的理由：
   The pathognomonic finding of reduced erythrocyte galactose-1-phosphate uridylyltransferase activity plus hypergalactosemia, jaundice, vomiting, feeding difficulties, hypotonia and seizures is the classic presentation of GALT-deficiency galactosemia; the HPO annotation for this entry is sparse (only 22 features, mostly metabolic/liver), which is why the ranking under-scored it, and the missing galactitol/cataract features are simply not documented in this patient.
```

一行一步，分四种：

| 标记 | 是什么 | 例子里的哪一行 |
|---|---|---|
| 第一行的 `[工具]` | 模型还没出场。先把 8,478 种病按症状排了名，前 30 名逐条核对 | 这时领先的三个病里**没有**半乳糖血症 |
| `[模型]` | 模型的一轮决定：这一轮要用哪些工具 | 第 1 轮它就决定按名字去找"半乳糖血症" |
| 其余的 `[工具]` | 工具查回来的结果，压成一句话 | `check_match: Galactosemia`：只对上 8 个表现里的 1 个 |
| `[答案]` | 最后的排名 | 第 1 名是半乳糖血症 |

**这道题正好说明 agent 比传统方法多出来的那部分能力。**

- 传统方法是"数对上了几条"。HPO 给半乳糖血症只登记了 22 个特征，这个病人的 8 个表现里只对得上 1 个（血里半乳糖高）。所以它被排到很后面，前 10 名里没有。看输出里的 `check_match: Galactosemia → explains 1 of 8 findings`，排在它前面的病都对上了 5 到 6 个
- agent 看到"GALT 酶活性降低"，认出这是半乳糖血症的决定性表现。第 1 轮它就按名字去找这个病（`find_disease`），第 2 轮核对，发现只对上 1 条，仍然把它放在第 1。它的理由（输出的最后一段）说得很清楚：GALT 酶活性降低加上高半乳糖血症是这个病的决定性表现；HPO 对这个病的记录很少（only 22 features），**所以排名把它低估了**（which is why the ranking under-scored it）。对上的条数少，是因为知识库记得少，不是因为病人不像。这种判断，数数的方法做不了

**花了多少**：这一次 10.6 秒，5 次模型调用，28,934 个 token，按高峰价约 $0.0020（半价时段再减半，不到 ¥0.01）。我跑过的几次都在 9 到 14 秒、4 到 8 次模型调用之间。大部分 token 命中了缓存：每一轮都要把前面的对话重新发一遍，重复的部分 DeepSeek 只收五十分之一的钱。

### 面试怎么说
> "The API streams the agent's steps as server-sent events. I didn't change the agent to make it observable: a LangChain callback handler, attached through a context variable for the duration of the call, relays its model and tool events — so the code that serves a request is the code that was evaluated. The same handler stops the run before its next model call when the deadline passes or the client disconnects."

---

## 第 4 步 · 模型出故障怎么办：自动降级

### 在干什么
模型用不了的时候，服务不报错。它改用"不需要模型的排名"来回答，并且在答案里明说：这是降级的。

### 背景
第 20 章大考考到一半，DeepSeek 账户欠费，接口开始回 402。当时的后果是"固定流程"被记成 0 分，我们为此加了一道"接口故障检查"。

**同样的故障发生在服务里，用户会看到什么？** 什么都不做的话，是一个报错页面。

只要依赖别人的接口，就一定会遇到它出故障：欠费、限流、超时、对方宕机。问题不是"会不会"，是"到时候怎么办"。

### 用了什么技术，为什么

打个比方：餐厅的煤气断了。不关门，改卖凉菜，并且在门口贴一张告示。

| 情况 | 服务怎么办 | 调用的人看到 |
|---|---|---|
| 模型正常 | agent 回答 | `mode: "agent"` |
| 没配密钥；接口报错（欠费、限流、超时、连不上）；agent 用完 8 轮还没给答案；超过 60 秒 | 用事先算好的"不用模型的排名"回答 | `mode: "fallback"`，`fallback_reason` 里写明原因 |
| **我们自己的代码出错** | **不降级**，回 500 和请求编号 | 一个明确的错误 |

**为什么自己的错误不降级**：降级会把问题藏起来。如果每个 bug 都被悄悄换成一个"还凑合"的答案，我们永远不知道代码坏了。别人的故障要兜住，自己的错误要露出来。

**怎么做到的**：每个请求一进来，先把"不用模型的排名"算好，只要 0.2 到 0.3 秒。它一物三用：是分段输出的第一条消息；是模型失败时的答案；并且不用等失败了才临时去算。

**降级以后有多准，是量过的。** 这一点最重要。两种降级用的系统，第 21 章都在留出集上考过：

| 840 道留出集的题 | 第 1 名 | 前 10 名 |
|---|---|---|
| agent | 42.4% | 62.0% |
| 降级：先排名再逐条核对 | 32.1% | 51.5% |
| 有基因，agent | 65.9% | 81.4% |
| 有基因，降级：按基因排名 | 59.5% | 80.0% |

所以"降级"不是一句空话。凉菜也是上过菜单、有人打过分的。

### 公司里什么情况怎么做
- **优雅降级**（graceful degradation）是标准做法。常见的层次：重试 → 换备用模型 → 换简单方法 → 实在不行才报错
- **每一层的质量都要事先量过**，不然降级以后是好是坏没人知道
- **熔断器**（circuit breaker）：对方连续失败很多次，就先不去试了，直接走降级，过一会儿再试
- **降级的次数是要盯着的指标**：突然变多，说明上游出事了

### 执行
再启动一个服务，这一次不给它模型密钥，问同一个病人（不花钱）：

In [ ]:
start(8023, with_model=False)
status, FALLBACK = ask("http://127.0.0.1:8023/v1/diagnose", example.request.model_dump(exclude_defaults=True))
print(f"\n状态码 {status}    回答方式：{FALLBACK['mode']}    原因：{FALLBACK['fallback_reason']}    用时 {FALLBACK['usage']['seconds']} 秒")
for s in FALLBACK["steps"]:
    print(f"   [{KIND[s['kind']]}] {s['title']}\n          {s['detail']}")
print("\n前 3 名：")
for d in FALLBACK["diagnoses"][:3]:
    m = d["evidence"]["match"]
    print(f"   {d['rank']}. {d['disease_name']}   对上 {len(m['explained'])}/{m['n_findings']} 个表现")
rank = next((d["rank"] for d in FALLBACK["diagnoses"] if d["disease_id"] == example.confirmed_disease_id), None)
print(f"\n论文确诊的病（{example.confirmed_disease_name}）：" + (f"排在第 {rank} 名" if rank else "不在前 10 名里"))
if AGENT_ANSWER is not None:
    agent_rank = next((d.rank for d in AGENT_ANSWER.diagnoses if d.disease_id == example.confirmed_disease_id), None)
    print(f"同一个病人，第 3 步 agent 的回答里：" + (f"排在第 {agent_rank} 名" if agent_rank else "不在前 10 名里"))

### 结果怎么看
- 状态码 200，不是报错。`mode` 是 `fallback`，原因写着 `no model API key is configured`（没有配置模型密钥）
- 用时零点几秒，没有调用模型，不花钱
- 第 1 名是 `3-Methylcrotonyl-CoA carboxylase 1 deficiency`（对上 6/8 个表现）。论文确诊的半乳糖血症**不在前 10 名里**
- 同一个病人，第 3 步里 agent 把它排在第 1

这就是降级的代价，在一道具体的题上的样子：服务没有倒，但这道题答不对了。整体上差多少，就是上面那张表：第 1 名从 42.4% 降到 32.1%。

> **一个真实踩过的坑：空的密钥**
>
> `.env.example` 里那一行是 `DEEPSEEK_API_KEY=`，等号后面是空的。照着复制成 `.env` 却忘了填的话，程序原来会拿着空密钥去请求 DeepSeek，得到一个让人摸不着头脑的 401。做这一步时才发现（要启动一个"没有密钥"的服务来演示）。现在空密钥按"没有密钥"处理，直接告诉你把密钥填进 `.env`；服务则以降级模式启动。

### 面试怎么说
> "Every request first computes the no-model ranking — about 300 ms — which is both the first streamed event and the fallback. If the model API errors, times out, or the agent doesn't answer, the response is that ranking with mode: fallback and the reason. I measured that path on the same held-out set: 32% top-1 against the agent's 42%, so degraded mode has a known quality. Our own bugs don't take that path: they return a 500 with the request id."

---

## 第 5 步 · 把门：哪些请求不接，哪些不能一直做下去

### 在干什么
在服务门口和服务里面，各放几道检查。

### 背景
回答一次要花钱（约 ¥0.02），占十来秒。没有限制的话：别人程序里一个写错的循环，一分钟发来几千个请求，账户就空了；一个卡住的请求会一直占着位置。

还有一种更隐蔽的风险：**悄悄答错**。下面第二行说的就是它。

### 用了什么技术，为什么

| 检查 | 防的是什么 | 对方收到什么 |
|---|---|---|
| 格式校验 | 乱七八糟的输入 | 422，并说明哪一项不对 |
| **编号必须真的存在** | 打错一位编号，被悄悄丢掉 | 422，并指出是哪个编号 |
| 同时最多回答 4 个 | 一下子来太多，钱和机器被拖垮 | 503，并附一句"5 秒后再试"（`Retry-After: 5`） |
| 密钥（可以不开） | 不认识的人来用 | 401 |
| 60 秒截止 | 一个请求没完没了 | 降级的答案 |
| 用户走了就停 | 没人看了还在花钱 | — |
| 请求编号 | 出了问题，对不上是哪一次 | 回答里、响应头里、日志里是同一个编号 |

**为什么不认识的编号要拒绝，不是跳过**：工具本身遇到不认识的编号是会跳过的。考试时没问题，题目都整理过。但对一个真病人，"悄悄少算一个症状，答案变了都没人知道"比"报错让你改一下"危险得多。所以在服务这一层，直接拒绝。

**为什么超过 4 个是直接拒绝，不是排队**：排队的话，对方不知道要等多久。直接说"现在忙，5 秒后再来"，对方的程序可以自己决定是重试还是放弃。行业里叫**削峰**（load shedding）。

**作废的编号是另一回事**：HPO 每次更新会作废一些旧编号，并指明换成哪个新的。这种情况不拒绝，自动换成现行的，并在回答里注明原来发的是哪个。

### 公司里什么情况怎么做
- **密钥和限流通常放在"网关"上统一做**（API gateway），所有服务共用一套
- **和业务有关的检查留在服务里**：比如"这个编号存不存在"，只有我们自己知道
- **每个限制都要有明确的返回码**：对方的程序是靠状态码决定下一步怎么办的

### 执行
对那个没有模型的服务（8023）发几个有问题的请求。全部不花钱：

In [ ]:
URL = "http://127.0.0.1:8023/v1/diagnose"
tries = [
    ("编号打错了一位", {"hpo_ids": ["HP:0001083", "HP:1234567"]}),
    ("想顺手发个名字", {"hpo_ids": ["HP:0001083"], "name": "张三"}),
    ("只发了遗传方式，没有症状", {"hpo_ids": ["HP:0000007"]}),
]
for label, body in tries:
    status, reply = ask(URL, body)
    detail = reply["detail"] if isinstance(reply["detail"], str) else reply["detail"][0]["msg"]
    print(f"{label}\n   → {status}  {detail}\n")

status, reply = ask(URL, {"hpo_ids": ["HP:0011148"]})
f = reply["findings"][0]
print(f"发了一个已经作废的编号 HP:0011148\n   → {status}  自动换成现行的 {f['hpo_id']}（{f['label']}），并注明原来发的是 {f['given_as']}\n")

status, reply = ask(URL, {"hpo_ids": ["HP:0001083"]}, headers={"X-Request-ID": "ward7-0042"})
print(f"自己带了一个请求编号 ward7-0042\n   → {status}  回答里的 request_id = {reply['request_id']}")

### 结果怎么看
- **编号打错一位** → 422，`not HPO terms in release 2026-09-01: HP:1234567`。它指出了是哪个编号、对照的是哪一版 HPO。另一个编号 `HP:0001083` 是对的，但整个请求仍然被拒绝：不会只拿一个症状去答
- **发名字** → 422，`Extra inputs are not permitted`
- **只有遗传方式** → 422，`none of the terms is a phenotypic finding`。`HP:0000007` 是"常染色体隐性遗传"，格式对、编号也存在，但它不是症状，没法拿去诊断
- **作废的编号** → 200，自动换成 `HP:0002121`（Generalized non-motor (absence) seizure），并注明原来发的是 `HP:0011148`
- **自己带的请求编号**原样回来了。医院那边的系统可以用自己的编号把一次请求从头跟到尾

并发上限、密钥、截止时间、用户走了就停，这四项在笔记里不方便演示（要同时发很多请求，或者故意让模型变慢），由自动测试覆盖：`tests/service/` 里有 39 个测试，全程用一个按剧本回答的假模型，不联网、不花钱。

### 面试怎么说
> "The tools tolerate an unknown HPO id by skipping it — fine in evaluation, dangerous for a real patient, because a typo silently changes the answer. At the service boundary an unknown term is a 422 that names it. Beyond a concurrency limit the service returns 503 with Retry-After instead of queueing, and the deadline and client disconnects are honoured between agent steps."

---

## 第 6 步 · 每次请求都留记录

### 在干什么
一次请求结束以后，能查到它的全部经过。

### 背景
上线以后一定会有人来问："上周二那个病人，为什么第 1 名是这个病？"没有记录，只能回答"不知道"。

前面每一次考试都有 MLflow 追踪（第 08、09 章）。服务用的是**同一套东西，同一种格式**。

### 用了什么技术，为什么

一次请求会在三个地方留下痕迹。它们各自记了什么、有没有病人的症状，是分开设计的：

| 记在哪 | 记了什么 | 里面有病人的症状吗 |
|---|---|---|
| **日志**：一行 JSON | 请求编号、回答方式、几秒、几次模型调用、多少 token、多少钱、追踪编号 | **没有**。只有"几个表现"这样的个数 |
| **MLflow 追踪** | 每一次模型调用、每一次工具调用的输入和输出 | 有（HPO 编号）。存在本机的 MLflow 里 |
| **发给 DeepSeek 的内容** | 症状、性别、候选基因 | 有，但没有身份信息：入口就不收（第 1 步） |

**为什么日志里不写症状**：日志通常会被集中收到别的系统里，能看到的人很多。而罕见病人的症状组合，几乎等于身份。所以日志只写个数，细节放在权限更严的追踪库里，两边用同一个编号对上。

**和考试用同一种格式，有什么好处**：第 20 章为"重放判断"写过一个工具，能从一次考试记录里读出每个候选病的证据。它可以原样用在线上的请求上（下面会试）。也就是说：**线上答错的请求，可以直接拿回来分析，甚至加进评测集。** 把线上的真实请求变回评测数据，是这一行（LLMOps）最核心的循环。

### 公司里什么情况怎么做
- **一个请求编号贯穿所有系统**：网关、服务、日志、追踪，都用它来对
- **追踪库有保存期限和访问权限**；医院里部署的话，追踪库要放在院内
- **换成公司的 MLflow 服务器只改一个设置**（`RAREDX_MLFLOW_TRACKING_URI`）。第 8 步的容器里就是这么做的

### 执行
先看日志里的那一行：

In [ ]:
LAST = AGENT_ANSWER.model_dump() if AGENT_ANSWER is not None else FALLBACK
port = 8022 if AGENT_ANSWER is not None else 8023
line = next(l for l in reversed((LOGS / f"{port}.log").read_text().splitlines())
            if '"event"' in l and LAST["request_id"] in l)
print(json.dumps(json.loads(line), indent=1, ensure_ascii=False))
print("\n这一行里有症状编号（HP:）吗？", "有" if "HP:" in line else "没有")

再用回答里的 `trace_id`，从 MLflow 里把这次请求的完整记录读出来：

In [ ]:
import mlflow
from collections import Counter
from raredx.evals.evidence import gather

mlflow.set_tracking_uri("sqlite:///mlflow.db")
trace = None
for _ in range(30):                     # 服务是在后台写记录的，稍等它写完
    try:
        trace = mlflow.get_trace(LAST["trace_id"])
    except Exception:
        trace = None
    if trace is not None and trace.data.spans:
        break
    time.sleep(1)

root = trace.data.spans[0]
print("追踪编号：", trace.info.trace_id)
print("最外层的一步：", root.name, "   对应的请求编号：", trace.info.client_request_id, "   回答方式：", trace.info.tags.get("mode"))
print("里面一共", len(trace.data.spans), "步：", dict(Counter(s.span_type for s in trace.data.spans)))
print("MLflow 统计的 token：", trace.info.token_usage)
print("回答里报的 token：   ", {k: LAST["usage"][k] for k in ("input_tokens", "cached_input_tokens", "output_tokens")})

evidence = gather(trace)
print(f"\n用第 20 章的重放工具读这次线上请求：读出 {len(evidence.candidates)} 个候选病的证据。前 3 个：")
for c in evidence.candidates[:3]:
    print(f"   {c.disease_name}：对上 {len(c.explained)}/{c.n_findings} 个表现，传统排名第 {c.phenotype_rank}，最后排在第 {c.agent_rank}")

### 结果怎么看
- **日志那一行**：有请求编号、`mode`、`n_findings: 8`（8 个表现，只有个数）、用时、模型调用次数、token、估算的花费、追踪编号。**没有任何 `HP:` 编号**
- **追踪**：最外层那一步叫 `diagnose`，里面有一百多步，其中 100 步左右是工具调用（`TOOL`），几步是模型调用（`CHAT_MODEL`）
  - （第 3 步没问模型的话，这里看到的是降级那次请求的记录，只有 50 多步。）工具调用为什么这么多：开局的"排名 + 核对前 30 名"是 31 次；agent 开局自己又做了一遍，再加 10 次查资料；最后给前 10 名各取一次证据，20 次。这些都在本机算，不花钱，加起来不到 1 秒
- **两边的 token 数对得上**：MLflow 自己统计的，和我们在回答里报的，是同一组数。花费是按这组数估的
- **第 20 章的重放工具能直接读这次请求**，读出了每个候选病的证据。线上和考试是同一种记录

### 面试怎么说
> "A request is one MLflow trace shaped exactly like an evaluation trace — question in, candidates out, every model and tool call a span — so the replay and error-analysis tools I built for evaluation runs work on production requests unchanged. The log line carries counts, cost and the trace id, and none of the patient's findings: a rare-disease phenotype is close to an identifier, and logs travel further than traces."

---

## 第 7 步 · 演示页面

### 在干什么
给不会写程序的人一个能点的页面：选一个病人，点一下，看它一步步查，看答案和证据。

### 背景
面试官、合作的医生不会用 `curl`。而且"看它一步步查"这件事，在页面上才直观。

### 用了什么技术，为什么

**Streamlit**：用 Python 写网页的工具。数据和 AI 团队做内部演示，基本都用它（另一个常见的是 Gradio）。

**页面自己不做任何判断。** 它通过上面那个接口去问，和任何别的程序一样。打个比方：页面是餐厅的"扫码点餐小程序"。后厨不知道、也不关心单子是从小程序来的，还是服务员递进来的。

为什么不让页面直接调用 agent：那样页面和 agent 就绑死了，别的团队想接入，还得再做一遍。**接口才是产品，页面只是它的一个用户。** 公司里正式产品的页面由前端团队用 React 之类的工具来写，调的是同一个接口。

页面长这样（示例 2，就是第 3 步那个病人）：

![演示页面](../img/demo.png)

- **左边**：现在是哪一版在答题，以及它在留出集上的成绩
- **中间上半**：病人的表现；点"Diagnose"以后，下面实时出现每一步
- **蓝色那一条**：论文确诊的病在这次答案里排第几，以及考试时这道题的情况
- **表格**：10 个候选病，每个对上了几个表现、agent 的理由
- **Evidence**：每个病的证据。第 1 名默认展开：对上了哪个表现、文献里多常见（2/3）、出处（PMID:35028268）、哪些表现它解释不了、病人缺少它的哪些常见特征

**5 个示例病人**，都来自留出集，按考试时的不同结果各挑了一个。右边一列是我这次在服务里实际跑出来的：

| 示例 | 论文确诊 | 考试时 | 这次在服务里 |
|---|---|---|---|
| 1 容易的 | Holt-Oram 综合征 | 所有系统都排第 1 | 第 1 |
| 2 agent 救回来的 | 半乳糖血症 | 传统排名前 10 里没有；agent 第 1 | 第 1（降级时不在前 10） |
| 3 基因让它升到第 1 | 范可尼贫血 C 型 | 只看症状在前 10、不是第 1；加上基因第 1 | 第 5 → 加基因后第 1 |
| 4 基因找回症状漏掉的 | Pitt-Hopkins 样综合征 1 | 只看症状没找到；加上基因第 1 | 没找到 → 加基因后第 1 |
| 5 谁也答不出的 | 僵硬皮肤综合征 | 没有系统找到 | 没找到 |

**为什么故意放一个答不出的**（示例 5）：HPO 对这个病的全部描述，只来自这个病人自己的那篇论文，遮掉以后知识库里就没有这个病了。演示里只放成功的例子，是在骗人。

**不往外发数据**：Streamlit 默认会向它的公司发送使用统计。`raredx demo` 启动时已经把它关掉了，和第 08 章关掉 MLflow 遥测是同一条原则。

**怎么打开**（在终端里，两个窗口）：

```
uv run raredx serve     # 窗口 1：服务
uv run raredx demo      # 窗口 2：页面。然后浏览器打开 http://127.0.0.1:8501
```

每点一次"Diagnose"约 ¥0.02。

### 公司里什么情况怎么做
- **内部演示用 Streamlit / Gradio，正式产品由前端团队写**，两者调同一个接口
- **演示用真实的、有代表性的例子，包括失败的**：只演示成功案例的东西，用户一上手就会失望

### 执行
页面上有个"按名字找术语"的搜索框，背后是接口 `/v1/terms`。再跑一下页面自己的自动测试（Streamlit 自带的测试工具会在后台"点"按钮；用的是假模型，不花钱）：

In [ ]:
run("curl -s 'http://127.0.0.1:8022/v1/terms?q=hearing&limit=3'")
run("uv run pytest tests/service/test_service_demo.py -q 2>&1 | grep -v INFO | tail -1")

### 结果怎么看
- 搜 `hearing`，第一个是 `Hearing impairment`（HP:0000365），后面是名字更长的
- 页面的 4 个测试通过：选示例并诊断、没有模型时显示降级、自己输入病人（包括打错编号时的提示）、服务没启动时的提示

> **一个真实踩过的坑：搜 hearing 找不到 Hearing impairment**
>
> 一开始搜索用的是底层库自带的功能，它区分大小写。搜 `hearing`，只能找到名字中间带小写 hearing 的术语（比如 Sensorineural **h**earing impairment），找不到以大写 H 开头的 **H**earing impairment，而它恰恰是最该排在第一个的。是测试先发现的。后来改成自己建一份不分大小写的索引。

### 面试怎么说
> "The demo is a thin Streamlit client of the same HTTP API another team would call. Its examples are held-out cases, sent with their own paper excluded — the conditions the reported numbers come from — and one of them is a case no system can answer, because a demo that only shows wins is misleading."

---

## 第 8 步 · 装进容器，让 CI 每次都检查

### 在干什么
把服务和它需要的一切打成一个"镜像"。换任何一台电脑，一条命令就能跑起来。

### 背景
"在我电脑上是好的"是软件行业最有名的一句话。要跑我们的服务，需要：Python 3.12；一百五十多个版本互相对得上的依赖包；HPO 的 4 个文件；一万多个病例。少一样，或者版本不对，就跑不起来，或者跑出不一样的答案。

### 用了什么技术，为什么

| 技术 | 是什么 | 打个比方 |
|---|---|---|
| **Docker 镜像** | 把系统、Python、依赖、代码、知识库全部打包在一起 | 一间标准化的实验室，整间搬走 |
| **Dockerfile** | 造这个镜像的配方，一步一步写明 | 实验室的装修图纸 |
| **多阶段构建** | 造的时候要用的工具（下载器、安装器），不带进成品 | 装修完把脚手架拆掉 |
| **Docker Compose** | 一个文件写明"同时启动哪几个容器、它们怎么互相找到" | 一张图纸上画了三间相邻的屋子 |

**镜像里有什么，没有什么**：

| | |
|---|---|
| 有 | 代码；锁定版本的依赖；HPO 2026-09-01；Phenopacket Store 0.1.27（造镜像时下载，逐个核对校验码，对不上就造不出来） |
| 没有 | **密钥**（运行时才给）；`.env`；测试和笔记用的工具；你电脑上的 `mlflow.db` |

**为什么把知识库也打进去**：一个答案取决于三样东西：代码、依赖、知识库。三样一起冻结，才说得清"这个答案是哪一版给的"。回答里的 `system.code` 就是造镜像时的那次提交。

**密钥怎么保证进不去**：`.dockerignore` 用的是**白名单**。先把所有文件挡在外面，再只放行需要的 5 样。以后项目里多出什么文件（比如又一个 `.env.local`），默认就进不去，不用记得去加一条"排除"。

**Compose 里的三个容器**，对应公司里的三个部署：

| 容器 | 干什么 | 地址 |
|---|---|---|
| `api` | 服务本身 | http://localhost:8000 （接口说明在 `/docs`） |
| `demo` | 演示页面，只通过接口去问 | http://localhost:8501 |
| `mlflow` | 追踪服务器，接收每次请求的记录 | http://localhost:5050 |

`api` 向 `mlflow` 汇报，只改了一个设置：`RAREDX_MLFLOW_TRACKING_URI=http://mlflow:5000`。第 08 章说过"以后换成公司的服务器，只改这一行"，这里兑现了。三个端口都只开在 `127.0.0.1` 上：只有这台电脑自己能访问。

**CI 每次都检查镜像**。每次提交代码，GitHub 自动做这几步：造镜像 → **不给密钥**启动 → 等它就绪 → 发一个问题（晶状体脱位、蜘蛛指、主动脉根部瘤，马凡综合征的经典三联征）→ 检查：是降级模式；前 10 名里有马凡综合征；版本号对得上。

不给密钥，走的就是降级那条路，所以这个检查**一分钱不花**，却把镜像、里面的知识库、整条请求链路都验了一遍。第 4 步做的降级，在这里又派上一次用场。

### 公司里什么情况怎么做
- **镜像推到镜像仓库，打上版本标签**；Kubernetes 从仓库拉镜像，启动多份，靠健康检查滚动更新
- **密钥由专门的密钥管理系统在运行时注入**，镜像和代码库里永远没有
- **镜像要扫描已知漏洞**，不以 root 身份运行（我们的容器用的是普通用户 `app`）

**这一章没有做的**，如实列出：推送到镜像仓库；Kubernetes；HTTPS；Prometheus 监控指标；网关上的限流。这些都是放到公司环境里才有意义的部分。

**镜像有 1.87 GB，偏大。** 主要是机器学习的库。公司里会把服务用不到的拆出去，比如第 18 章那套文本向量化的库：最终系统没用它，但它还在依赖里。

### 执行
下面这一格需要 Docker 在运行，并且已经造好镜像（`docker compose build`，第一次约 5 分钟）。没有的话它会说明并跳过。它启动一个**没有密钥**的容器，问一个问题，再把容器删掉，不花钱：

In [ ]:
docker_up = subprocess.run("docker info", shell=True, capture_output=True).returncode == 0
have_image = docker_up and subprocess.run("docker image inspect raredx:local", shell=True, capture_output=True).returncode == 0
if not docker_up:
    print("Docker 没有在运行，这一格跳过。\n要试的话：先在终端运行 colima start（或打开 Docker Desktop），再重新运行这一格。")
elif not have_image:
    print("还没有镜像，这一格跳过。\n要试的话：在终端运行 docker compose build（第一次约 5 分钟），再重新运行这一格。")
else:
    run("docker images raredx:local --format '镜像 {{.Repository}}:{{.Tag}}    大小 {{.Size}}'")
    run('''docker run --rm raredx:local sh -c 'echo "运行身份: $(id -un)"; echo "镜像里的东西: $(ls /app | tr "\\n" " ")"; echo "环境变量里的密钥: $(env | grep -ciE "deepseek|api_key") 个"' ''')
    run("docker rm -f raredx-ch22 >/dev/null 2>&1; docker run -d --name raredx-ch22 -p 127.0.0.1:8032:8000 raredx:local")
    for _ in range(90):
        try:
            urllib.request.urlopen("http://127.0.0.1:8032/readyz", timeout=1)
            break
        except Exception:
            time.sleep(1)
    status, reply = ask("http://127.0.0.1:8032/v1/diagnose", {"hpo_ids": ["HP:0001083", "HP:0001166", "HP:0002616"]})
    print(f"容器里的服务回答：状态码 {status}    方式 {reply['mode']}（{reply['fallback_reason']}）    代码版本 {reply['system']['code']}")
    for d in reply["diagnoses"][:4]:
        print(f"   {d['rank']}. {d['disease_name']}")
    run("docker rm -f raredx-ch22")

最后，把这本笔记启动的两个服务停掉：

In [ ]:
stop(8022)
stop(8023)
print("两个服务都停了。日志在临时文件夹里：", LOGS)

### 结果怎么看
- 镜像 `raredx:local`，1.87 GB
- 容器里：运行身份是 `app`（不是 root）；镜像里只有 `data`、`evals`、`state` 这几样（代码和依赖在一个隐藏的 `.venv` 文件夹里）；环境变量里的密钥是 **0 个**
- 容器里的服务回答了：降级模式，原因是没有密钥；前 4 名里有 `Marfan syndrome`（马凡综合征）；代码版本是造镜像时的那次提交
- 这和 CI 里做的是同一个检查

要把三个容器一起跑起来（带上 `.env` 里的密钥，agent 可用）：

```
docker compose up --build        # 第一次约 5 分钟
```

然后浏览器打开 http://localhost:8501 。停掉：`docker compose down`。

> **一个真实踩过的坑：这台电脑的 Docker 少一个组件**
>
> 网上推荐的 Dockerfile 写法里有一种加速用的指令（`RUN --mount=...`），它需要一个叫 buildx 的组件。这台电脑上没装，一用就报错：`the --mount option requires BuildKit`。所以配方改成了不依赖它的写法，功能一样，只是重复造镜像时慢一点。教训：配方要在最普通的环境里也能用。

> **另一个真实踩过的坑：好好的容器显示"不健康"**
>
> 三个容器用的是同一个镜像。镜像自带的健康检查问的是 8000 端口，那是服务的端口。结果演示页面的容器明明好好的（它在 8501 端口），却一直显示"正在启动"，再等下去会被判成"不健康"。后来在 Compose 文件里给页面和 MLflow 各写了对应自己端口的检查。

### 面试怎么说
> "The image holds the code, the locked dependencies and the pinned knowledge, downloaded and checksum-verified at build time — everything an answer depends on, as one versioned artifact — and no secret: .dockerignore is an allow-list and the key arrives in the environment. CI builds the image, starts it without a key and asks it a question over HTTP; because that exercises the fallback path, the whole request path is tested on every commit for free."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 定接口 | FastAPI、pydantic、自动生成的接口说明 | 团队之间靠接口协作；不该收的数据在入口就不收 |
| 启动服务 | uvicorn、存活检查、就绪检查、版本信息 | 能被别的系统调用，能被容器平台管理 |
| 实时看每一步 | SSE、LangChain 回调（不改 agent） | 用户等得住，也信得过 |
| 降级 | 事先算好不用模型的答案；降级的质量量过 | 上游出故障，服务不倒 |
| 把门 | 校验、并发上限、密钥、截止时间、取消 | 不白花钱，不被拖垮，不悄悄答错 |
| 留记录 | 每次请求一条 MLflow 追踪 + 一行不含症状的日志 | 事后能查；线上请求能变回评测数据 |
| 演示页面 | Streamlit，只通过接口去问 | 不会写程序的人也能试 |
| 容器 | Dockerfile、Compose、CI 里启动容器提问 | 换台电脑能跑；坏的镜像合并不进去 |

**做了多少**：服务的代码约 1,500 行（含演示页面），39 个测试（全程用假模型，不花钱），整个项目的测试从 234 个到 277 个。

**没有做的**：把病历里的文字自动转成 HPO 编号（现在只收编号）；和通用诊断工具、通用大模型的比较；Kubernetes、HTTPS、监控指标这些要放到公司环境里才有意义的部分。

## 一句话

**考的是什么，上线的就是什么**：服务把请求原样交给考过的那个 agent，自己只在周围加上接口、检查、记录和兜底。

## 面试题

1. **How did you take the agent from an evaluation harness to a service without invalidating the evaluation?**
   > "The service calls the evaluated agent's predict unchanged and routes each request to the configuration evaluated for that kind of input. Observability is attached from outside through a LangChain callback handler, so I didn't touch the agent; `git diff` against the evaluated tag is empty for the agent, tools and masking code. The response reports the model, prompt version, knowledge release and commit."

2. **What happens when the model API is down?**
   > "The request still succeeds. The no-model ranking is computed first for every request, and it becomes the answer if the API errors, times out, the agent doesn't answer, or the deadline passes — marked mode: fallback with the reason. I measured that path on the held-out set, so I can say what degraded mode costs: ten points of top-1. Bugs in our own code are not caught by it."

3. **How do you handle patient data?**
   > "By not accepting it. The schema takes HPO ids, sex, gene symbols and PubMed ids, pattern-checked, extra fields forbidden — no free text. Logs carry counts and ids only. Traces hold the phenotype and stay in the MLflow store, which in a hospital would sit inside its network. The model provider sees coded phenotype terms and nothing identifying."

4. **How do you know the container works?**
   > "CI builds the image, starts it without a model key, waits for readiness and posts a textbook case. Without a key the service takes its fallback path, so the image, the knowledge baked into it and the whole HTTP path are checked on every commit without a paid call."

5. **What would you add before real production use?**
   > "Authentication and rate limiting at a gateway, metrics and alerts — especially on the fallback rate — a registry and orchestrated rollout, a circuit breaker in front of the model, and an evaluation of the full patient bank before switching it on. And the step I left out on purpose: turning free-text notes into HPO terms, which needs its own evaluation."

**下一步**（还没开始，等你决定）：和通用工具、通用大模型在同一份 840 道题上比较。这一步要花钱，开始前先定预算。